In [10]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# --------------------------------------------------
# 1. Set MLflow Experiment
# --------------------------------------------------

mlflow.set_experiment("AML_Assignment2")

# --------------------------------------------------
# 2. Load Data
# --------------------------------------------------

train = pd.read_csv(r"C:\Users\Ahana\Documents\AML_Assignment2\data\train.csv")
val = pd.read_csv(r"C:\Users\Ahana\Documents\AML_Assignment2\data\validation.csv")

TARGET = "label"
TEXT_COL = "text"   # <-- Change if your text column name is different

X_train = train[TEXT_COL]
y_train = train[TARGET]

X_val = val[TEXT_COL]
y_val = val[TARGET]

# Encode target if necessary
if y_train.dtype == "object":
    le = LabelEncoder()
    y_train = le.fit_transform(y_train)
    y_val = le.transform(y_val)

# --------------------------------------------------
# 3. Define Training Function
# --------------------------------------------------

def train_and_log_model(model, model_name):

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000)),
        ("clf", model)
    ])

    with mlflow.start_run(run_name=model_name):

        # Train
        pipeline.fit(X_train, y_train)

        # Predict probabilities
        probs = pipeline.predict_proba(X_val)[:, 1]

        # Compute AUCPR
        aucpr = average_precision_score(y_val, probs)

        # Log metric
        mlflow.log_metric("AUCPR", aucpr)

        # Log model
        mlflow.sklearn.log_model(pipeline, name = "model", serialization_format="skops")

        print(f"{model_name} AUCPR: {aucpr:.4f}")


# --------------------------------------------------
# 4. Train 3 Benchmark Models
# --------------------------------------------------

train_and_log_model(
    LogisticRegression(max_iter=1000),
    "Logistic_Regression"
)

train_and_log_model(
    RandomForestClassifier(n_estimators=100, random_state=42),
    "Random_Forest"
)

train_and_log_model(
    DecisionTreeClassifier(random_state=42),
    "Decision_Tree"
)


2026/02/11 15:46:28 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Ahana\AppData\Local\Temp\tmpvykjh5y1\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 


Logistic_Regression AUCPR: 0.9725


2026/02/11 15:46:37 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Ahana\AppData\Local\Temp\tmpndfaoh8t\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 


Random_Forest AUCPR: 0.9826


2026/02/11 15:46:45 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Ahana\AppData\Local\Temp\tmpq4hj3jqa\model\model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.1', 'skops==0.13.0']. Set logging level to DEBUG to see the full traceback. 


Decision_Tree AUCPR: 0.7576
